# Per-perturbation neural bridge: leave-one-batch-out evaluation

Set `n_validation_batches` below to choose how many distinct batches to evaluate, one at a time. Train a fresh per-perturbation neural bridge and the linear baselines for each selected batch, using **every other batch** for training. This gives `n_validation_batches` independent fits per model; batches held out in other runs remain available for training.

Only control inputs are standardised, using that run's training-control statistics. Targets and predictions stay in the dataset's existing expression units. The neural model's inner cross-fitting remains enabled.

Each held-out batch gets its own metric plots. The x-axis counts training batches containing that perturbation; each dot is one perturbation/model score, with no averaging across held-out batches.

In [3]:
import sys
from datetime import datetime

import torch
from IPython.display import Image, display
from omegaconf import OmegaConf

from genetic_perturbation_playground.data.data_loader import DATA_DIR, load_dataset
from genetic_perturbation_playground.utils.plotting import report_by_training_batches

sys.path.insert(0, str(DATA_DIR.parent / "scripts"))
from train_batch_transfer_expression import cross_validate

n_validation_batches = 1  # n separate runs, each holding out one batch

cfg = OmegaConf.merge(
    OmegaConf.load(DATA_DIR.parent / "configs" / "train_batch_transfer_expression.yaml"),
    {
        "name": "perx_leave_one_batch_out",
        "n_validation_batches": n_validation_batches,
        "seed": 0,
        "top_k": 20,
        "bridges": ["neural"],
        "bridge_reg": 0.0001,
        "neural": {
            "per_perturbation": True,
            "weight_decay": 0.0001,
            "n_features_w": 8,
            "width_w": 8,
            "depth": 2,
            "lr": 0.00075,
            "n_steps": 400,
            "cross_fit_folds": 5,
            "device": "cuda" if torch.cuda.is_available() else "cpu",
        },
    },
)
output_dir = DATA_DIR.parent / "outputs" / f"{cfg.name}_{datetime.now():%Y%m%d_%H%M%S}"

## Data and control split

The script selects the validation batches and splits each batch's controls into two halves once. Models use the seen half as inputs; scoring uses the other half as the control baseline. All perturbed cells in the remaining batches enter training.

In [4]:
adata = load_dataset()
n_batches = adata.obs["batch"].nunique()
print(f"{cfg.n_validation_batches} separate runs: one held-out batch and {n_batches - 1} training batches each")

Loaded /nfs/ghome/live/adhir/genetic_playground/data/replogle_k562_preprocessed.h5ad
  310,385 cells  ×  8,563 genes  |  2,058 perturbations
1 separate runs: one held-out batch and 47 training batches each


## Train and score

Reuse `train_batch_transfer_expression.cross_validate` for the requested number of leave-one-batch-out fits, input scaling, and scoring. The final model from each run is evaluated on its single held-out batch; no checkpoint is selected using that batch.

`scores` keeps every (held-out batch, perturbation, metric, model) result and its `n_training_batches` count. The neural model is labelled `neural bridge` by the shared script; `per_perturbation=True` selects the per-x implementation.

In [ ]:
_, per_batch, scores = cross_validate(adata, cfg)
output_dir.mkdir(parents=True, exist_ok=True)
scores.to_csv(output_dir / "scores.csv", index=False)
print(f"Saved scores to {output_dir}")

validation 1/1: holding out 13


## Metrics by training-batch count

Reuse `report_by_training_batches` to save and display a separate figure for each held-out batch and metric (MSE, Pearson, and top-k Pearson). Each batch folder also contains the plotted rows as `per_perturbation.csv`.

The x-axis is the number of **training batches with observed cells for that perturbation**, excluding the held-out batch. It is not the total number of batches used to train the shared neural feature map. The figures use individual perturbation scores without averaging.

In [ ]:
plot_dir = output_dir / "plots" / "by_training_batches"
report_by_training_batches(scores, plot_dir)
for path in sorted(plot_dir.glob("batch_*/*.png")):
    display(Image(filename=str(path)))

## Summary for each held-out batch

This table averages over perturbations within each held-out batch, keeping batches separate. The plots above retain the individual scores.

In [ ]:
per_batch.pivot(index=["batch", "metric"], columns="model", values="value").style.format("{:.4g}")